# Experiment 3: Shift-Equivariant ViT ($\mathbb{Z}^2$-Equivariant Encoder)

## Symmetry Group

The discrete translation group $(\mathbb{Z}^2, +)$ acts on an $H \times W$ image by cyclic pixel shifts:

$$(t_{a,b} \cdot x)[i, j, c] = x[(i + a) \bmod H,\; (j + b) \bmod W,\; c]$$

## Architecture (Rojas-Gomez et al., CVPR 2024)

Four adaptive polyphase modules enforce exact $\mathbb{Z}^2$-equivariance:

- **A-token**: Per-sample polyphase offset selection
- **A-WSA**: Adaptive window self-attention partition
- **A-PMerge**: Polyphase downsampling
- **A-RPE**: Circular relative position encoding

No extra learnable parameters are introduced. The encoder satisfies:

$$\varphi_\theta(t_{a,b} \cdot x) = \varphi_\theta(x) \quad \forall (a, b) \in \mathbb{Z}^2$$

## Fusion and Loss

$$h_\psi(z_1, z_2) = W_2\,\mathrm{ReLU}\!\bigl(W_1\,[|z_1 - z_2|\;\|\;z_1 \odot z_2] + b_1\bigr) + b_2$$

$$\mathcal{L} = \mathcal{L}_{\mathrm{BCE}} + \lambda\,\mathcal{L}_{\mathrm{contr}}^{L^2}$$

## Runs

| Run | Augmentation | $\lambda$ |
|-----|-------------|----------|
| `shift_eq_no_reg` | Block P only | 0.0 |
| `shift_eq_reg` | Block P only | 0.1 |

**Key hypothesis**: Shift-invariant encoder should excel on cropping/recentering plagiarism, but may not help with rotations.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import torch
import yaml
import logging
import matplotlib.pyplot as plt

from src.utils.seed import set_seed
from src.data_module.augmentations import build_train_transform, build_val_transform
from src.data_module.coco_dataset import build_coco_dataloaders
from src.data_module.domainnet_dataset import DomainNetEvalDataset
from src.encoders import EncoderFactory
from src.losses.composite import CompositeLoss
from src.models.siamese import SiameseNet
from src.training.trainer import Trainer
from src.validation.evaluator import DomainNetEvaluator
from src.validation.tsne import compute_tsne_embeddings, plot_tsne
from torch.utils.data import DataLoader

logging.basicConfig(level=logging.INFO)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
with open('../configs/default.yaml') as f:
    cfg = yaml.safe_load(f)

set_seed(cfg['seed'])
train_tfm = build_train_transform(cfg)
val_tfm = build_val_transform(cfg)
dataloaders = build_coco_dataloaders(cfg, train_tfm, val_tfm)

In [ ]:
histories = {}
models = {}

for run_name, lam in [('shift_eq_no_reg', 0.0), ('shift_eq_reg', 0.1)]:
    print(f'\n{"="*60}')
    print(f'Run: {run_name}, lambda={lam}')
    print(f'{"="*60}')
    
    set_seed(cfg['seed'])
    encoder = EncoderFactory('shift_eq_vit', freeze=False)
    model = SiameseNet(encoder, hidden_dim=512, dropout=0.1)
    
    criterion = CompositeLoss(
        bce_weight_pos=0.3, bce_weight_neg=0.7,
        contrastive_margin=1.0, lambda_reg=lam,
    )
    optimizer = torch.optim.AdamW(model.head.parameters(), lr=1e-3, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[5, 10], gamma=0.5)
    
    trainer = Trainer(
        model=model, criterion=criterion, optimizer=optimizer,
        scheduler=scheduler, device=device,
        checkpoint_dir=f'../outputs/checkpoints/{run_name}',
    )
    histories[run_name] = trainer.fit(
        dataloaders['train'], dataloaders['val'],
        num_epochs=cfg['training']['num_epochs'],
        run_name=run_name,
    )
    models[run_name] = model

## DomainNet Evaluation

In [ ]:
domainnet_ds = DomainNetEvalDataset(
    root=cfg['data']['domainnet_root'],
    domains=cfg['data']['domainnet_domains'],
    pairs_per_domain=cfg['data']['pairs_per_domain'],
)

for run_name, model in models.items():
    print(f'\n--- {run_name} ---')
    evaluator = DomainNetEvaluator(model=model, dataset=domainnet_ds, device=device)
    report = evaluator.run()
    print(json.dumps(report, indent=2))

## t-SNE: Shift-Equivariant ViT Embeddings

In [ ]:
last_model = list(models.values())[-1]
tsne_loader = DataLoader(domainnet_ds, batch_size=32, shuffle=False, num_workers=2)
coords, domains = compute_tsne_embeddings(last_model, tsne_loader, device=device)
fig = plot_tsne(coords, domains, title='t-SNE: Shift-Eq ViT ($\\mathbb{Z}^2$)',
                save_path='../outputs/figures/shift_eq_vit_tsne.pdf')
plt.show()

## Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, hist in histories.items():
    epochs = [h['epoch'] for h in hist]
    axes[0].plot(epochs, [h['train']['loss'] for h in hist], label=f'{name} (train)')
    axes[1].plot(epochs, [h['val']['loss'] for h in hist], label=f'{name} (val)')
axes[0].set_title('Train Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')
axes[1].set_title('Val Loss'); axes[1].legend(); axes[1].set_xlabel('Epoch')
fig.tight_layout()
fig.savefig('../outputs/figures/shift_eq_curves.pdf', dpi=300)
plt.show()